# MobileNetV2 embeddings + hybrid model E (subject-independent protocol, all DMD participants)

Computes the MobileNetV2 embeddings of all face crops (`results/emb_all_v2.npy`, used by every later step) and a first LOPO comparison.

**Protocol:**
1. Leak-free normalisation: from the features and embeddings of each recording we subtract the median over its first 10 s (no labels used), as in `04b_baseline_m1_honest.py`.
2. Frames without a face stay in the test set and get a fixed answer (attention → not_looking_road, drowsy → alert), the same for all variants.

**Variants:**
- **A**: face geometry (EAR, MAR, head angles + 5-frame window), random forest; should match `04b`;
- **C**: raw embeddings, logistic regression (disabled below);
- **D**: calibrated embeddings, logistic regression;
- **E**: hybrid: calibrated embeddings (PCA to 32) + face geometry, random forest.

**Inputs on Drive** (`MyDrive/driver_project/`): `dmd_for_colab.zip` with `labels.csv` and the face crops `frames/` from `02_extract_features.py`.

**Run:** Runtime → T4 GPU, set `TASK`, Run all. For the other task change `TASK` and run again; the embeddings are not recomputed (cached on Drive).

In [ ]:
import os, glob, subprocess
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## Settings
Only `TASK` changes. The window and calibration were fixed in advance, not tuned on the test set.

In [ ]:
TASK = "attention"   # "drowsy" or "attention"

WINDOW = 5        # frames (1 s at 5 fps)
CALIB_SEC = 10    # calibration seconds at the start of each recording
FPS = 5

ZIP_PATH  = "/content/drive/MyDrive/driver_project/dmd_for_colab.zip"
OUT_DIR   = "/content/drive/MyDrive/driver_project/results"
DATA_DIR  = "/content/data"
EMB_CACHE = f"{OUT_DIR}/emb_all_v2.npy"   # embeddings of all frames with a face (shared by both tasks)

## Data
Mount Drive and unpack the archive (78,359 face crops for all 14 participants).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(OUT_DIR, exist_ok=True)

if not os.path.exists(f"{DATA_DIR}/labels.csv"):
    os.makedirs(DATA_DIR, exist_ok=True)
    subprocess.run(["unzip", "-q", ZIP_PATH, "-d", DATA_DIR], check=True)

print(len(glob.glob(f"{DATA_DIR}/frames/*/*.jpg")), "images")

## Frame table
The k-th row with a face in a recording corresponds to file `k.jpg` in its folder (as saved by `02_extract_features.py`).

In [ ]:
BASE = ["ear", "mar", "yaw", "pitch", "roll"]
TASKS = {
    "drowsy":    ("drowsy_label",    ["alert", "eyes_closed", "yawning"]),
    "attention": ("attention_label", ["looking_road", "not_looking_road"]),
}
NOFACE_RULE = {"drowsy": "alert", "attention": "not_looking_road"}
label_col, classes = TASKS[TASK]

lab = pd.read_csv(f"{DATA_DIR}/labels.csv", keep_default_na=False, na_values=[""], low_memory=False)
for c in BASE:
    lab[c] = pd.to_numeric(lab[c], errors="coerce")
lab["face_found"] = pd.to_numeric(lab["face_found"], errors="coerce").fillna(0)

face = lab[lab["face_found"] == 1].sort_values(["video_id", "frame_idx"]).copy()
face["k"] = face.groupby("video_id").cumcount()
face["path"] = f"{DATA_DIR}/frames/" + face["video_id"] + "/" + face["k"].map("{:06d}.jpg".format)
face["eidx"] = np.arange(len(face))
missing = int((~face["path"].map(os.path.exists)).sum())
print("Frames with a face:", len(face), "| files not found:", missing)
assert missing == 0, "Some images are missing: check the zip on Drive"
print("People:", sorted(lab["person"].unique()))

## MobileNetV2 embeddings
One pass over all frames with a face (no training, ~5 min on a T4). The result is cached on Drive.

In [ ]:
if os.path.exists(EMB_CACHE) and len(np.load(EMB_CACHE, mmap_mode="r")) == len(face):
    E_all = np.load(EMB_CACHE)
    print("Loaded from cache:", E_all.shape)
else:
    base = tf.keras.applications.MobileNetV2(include_top=False, weights="imagenet",
                                             input_shape=(224, 224, 3), pooling="avg")
    def load(path):
        img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
        img = tf.image.resize(img, (224, 224))
        return tf.keras.applications.mobilenet_v2.preprocess_input(tf.cast(img, tf.float32))
    ds = (tf.data.Dataset.from_tensor_slices(face["path"].values)
          .map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(128).prefetch(tf.data.AUTOTUNE))
    E_all = base.predict(ds, verbose=1).astype("float32")
    np.save(EMB_CACHE, E_all)
    print("Computed:", E_all.shape)

## Features and calibration
Calibration and windows use all frames of a recording with a face, including frames without the task label, as the system would in a car.

In [ ]:
def add_window_features(df, k):
    df = df.sort_values(["video_id", "frame_idx"]).copy()
    g = df.groupby("video_id")
    feats = list(BASE)
    if k > 1:
        for c in ["ear", "mar"]:
            for name in ["mean", "min", "max", "std"]:
                col = f"{c}_{name}{k}"
                df[col] = g[c].transform(lambda s, n=name: getattr(s.rolling(k, min_periods=1), n)())
                feats.append(col)
        for c in ["yaw", "pitch"]:
            for name in ["mean", "std"]:
                col = f"{c}_{name}{k}"
                df[col] = g[c].transform(lambda s, n=name: getattr(s.rolling(k, min_periods=1), n)())
                feats.append(col)
        wcols = [x for x in feats if x not in BASE]
        df[wcols] = df[wcols].fillna(0)
    return df, feats

n_cal = int(round(CALIB_SEC * FPS))
f = face.copy()
f[BASE] = f.groupby("video_id")[BASE].ffill(limit=3)
f = f.dropna(subset=BASE).sort_values(["video_id", "frame_idx"])
is_cal = (f.groupby("video_id").cumcount() < n_cal).values

# face features: minus the median of the first 10 s of the recording
ref = f[is_cal].groupby("video_id")[BASE].median()
f[BASE] = f[BASE].values - ref.reindex(f["video_id"]).values
f, feats = add_window_features(f, WINDOW)   # row order unchanged (already sorted)

# embeddings: raw and with the same calibration
E = E_all[f["eidx"].values]
Ec = E.copy()
vids = f["video_id"].values
for v in np.unique(vids):
    m = vids == v
    Ec[m] -= np.median(E[m & is_cal], axis=0)

# labelled frames of this task only
mask = f[label_col].isin(classes).values
fl, El, Ecl = f[mask], E[mask], Ec[mask]
X_A = fl[feats].values
labeled = lab[lab[label_col].isin(classes)]
noface = labeled[~labeled.index.isin(fl.index)]   # no face, or features could not be computed

cnt = pd.concat([fl, noface]).groupby(["person", label_col]).size().unstack(fill_value=0)
cnt["of which no face"] = noface.groupby("person").size().reindex(cnt.index, fill_value=0)
print(cnt)
import gc
del E_all, E, Ec        # only El and Ecl (labelled frames) are needed from here on
gc.collect()

## Training: one person in test, the rest in training
Everything fitted to data (scaling, PCA, models) is fitted on the training people only.

In [ ]:
def m_rf():
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=3, class_weight="balanced",
                                  random_state=0, n_jobs=-1)

def m_lr():
    return make_pipeline(StandardScaler(),
                         LogisticRegression(C=0.1, max_iter=3000, class_weight="balanced"))

def b_A(tr, te): return X_A[tr], X_A[te]
def b_C(tr, te): return El[tr], El[te]
def b_D(tr, te): return Ecl[tr], Ecl[te]
def b_E(tr, te):
    sc = StandardScaler().fit(Ecl[tr])
    pca = PCA(32, random_state=0).fit(sc.transform(Ecl[tr]))
    z = lambda idx: pca.transform(sc.transform(Ecl[idx]))
    return np.hstack([z(tr), X_A[tr]]), np.hstack([z(te), X_A[te]])

configs = {
    "A_face_features":       (b_A, m_rf),
    ##"C_emb_raw":          (b_C, m_lr),
    "D_emb_calibrated":   (b_D, m_lr),
    "E_hybrid (D + A)":      (b_E, m_rf),
}

y_face, p_face = fl[label_col].values, fl["person"].values
persons = sorted(set(p_face) | set(noface["person"]))
results = {}
for name, (build, make_model) in configs.items():
    print("running:", name)
    pred = np.empty(len(fl), dtype=object)
    for p in persons:
        te = p_face == p
        if not te.any():
            continue
        Xtr, Xte = build(~te, te)
        model = make_model().fit(Xtr, y_face[~te])
        pred[te] = model.predict(Xte)
    results[name] = pred

## Results
Macro-F1 per person (including frames without a face), mean and SD over people. For drowsiness also the share of yawning frames found with and without a hand, and the number of false yawn frames.

In [ ]:
y_all = np.concatenate([y_face, noface[label_col].values])
p_all = np.concatenate([p_face, noface["person"].values])
yawn_all = np.concatenate([fl["yawning"].astype(str).values, noface["yawning"].astype(str).values]) \
    if "yawning" in fl.columns else None
rule = np.array([NOFACE_RULE[TASK]] * len(noface), dtype=object)

rows, all_preds = [], {}
for name, pred in results.items():
    pr = np.concatenate([pred, rule])
    all_preds[name] = pr
    row, f1s = {"config": name}, []
    for p in persons:
        m = p_all == p
        f1 = f1_score(y_all[m], pr[m], labels=sorted(set(y_all[m])), average="macro", zero_division=0)
        row[p] = round(f1, 3); f1s.append(f1)
    row["mean"] = round(float(np.mean(f1s)), 3)
    row["std_people"] = round(float(np.std(f1s, ddof=1)), 3)
    if TASK == "drowsy" and yawn_all is not None:
        is_y = y_all == "yawning"
        for t, short in [("Yawning with hand", "yawn_with_hand_%"), ("Yawning without hand", "yawn_no_hand_%")]:
            mt = is_y & (yawn_all == t)
            row[short] = round(100 * float((pr[mt] == "yawning").mean()), 1) if mt.any() else np.nan
        row["false_yawn_frames"] = int(((pr == "yawning") & ~is_y).sum())
    rows.append(row)

table = pd.DataFrame(rows)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print(table.to_string(index=False))
print(f"\nFrames without a face in test: {len(noface)} (same rule '{NOFACE_RULE[TASK]}' for all variants)")

## Save

In [ ]:
table.to_csv(f"{OUT_DIR}/summary_m0b_honest_{TASK}.csv", index=False)
out = pd.concat([fl[["video_id", "frame_idx", "person", label_col]],
                 noface[["video_id", "frame_idx", "person", label_col]]], ignore_index=True)
out["face"] = np.r_[np.ones(len(fl), int), np.zeros(len(noface), int)]
for name, pr in all_preds.items():
    out[name[:1]] = pr
out.to_csv(f"{OUT_DIR}/preds_m0b_honest_{TASK}.csv", index=False)
print("Saved to", OUT_DIR)

In [ ]:
from scipy.stats import wilcoxon
a = table.set_index("config").loc["A_face_features", persons].astype(float)
e = table.set_index("config").loc["E_hybrid (D + A)", persons].astype(float)
print("E - A:", round((e - a).mean(), 4), "| better for", (e > a).sum(), "of", len(persons))
print(wilcoxon(e, a))

In [ ]:
d = pd.read_csv(f"{OUT_DIR}/summary_m0b_honest_drowsy.csv").set_index("config")
pp = [c for c in d.columns if c.startswith("g")]
a = d.loc["A_face_features", pp].astype(float); e = d.loc["E_hybrid (D + A)", pp].astype(float)
print("E - A:", round((e - a).mean(), 4), "| better for", (e > a).sum(), "of", len(pp))
print(wilcoxon(e, a))